# Week 8 workshop · Verify the Abelian sandpile

<span class="workshop-download-enabled" aria-hidden="true"></span>

# Context


**Canonical model:** the Abelian sandpile.

**Modelling practice:** check supplied code and decide how many runs are enough.

This week we return to the ladder of abstraction. Up, down, up, down, .... Pause the simulation to inspect individual topplings, then build back up to avalanche measurements and ensemble summaries. We need these moves whenever a model or its code is unfamiliar.

By the end, you should be able to:

1. check that reused code follows the model rules and records the quantities you need;
2. decide how many independent runs are enough for the quantity and precision you need.

# Specify the model

## Model specification

| Component | Workshop model |
|---|---|
| world | finite $L\times L$ square lattice; neighbours above, below, left and right |
| state | non-negative integer load $z_{ij}$ at each site |
| threshold | $z_{ij}\geq4$ is unstable |
| toppling | subtract four grains; send one to each of the four neighbours |
| drive | choose a site uniformly at random and add one grain, after complete relaxation |
| relaxation | each site unstable at the start of a parallel step topples once; repeat until all sites are stable |
| boundary | grains sent beyond the lattice are lost (sand falls off the table) |
| trial outputs | total topplings $S_n$, distinct toppled sites $A_{\mathrm{av},n}$, parallel steps $T_n$ |

Trial $n$ is one addition followed by complete relaxation. The resulting avalanche is one event; each toppling is a redistribution within it. With no toppling, $S_n=A_{\mathrm{av},n}=T_n=0$. Within a trial, $k$ counts parallel relaxation steps; $S_n$ sums their topplings. We omit $n$ in distributions. Reversing the order of topplings within each parallel step gives the same final pile and measurements.

Use integer piles. An initial pile supplied to `run_sandpile` must be stable and have shape $(L,L)$; choose `burn_in` between zero and `additions`.

# Verify the implementation

Read the functions. What does each receive, return and change in place? Find where each model rule is implemented.

Identify how each starting-pile option is made, how addition sites are chosen and where grains can leave the lattice.

In [40]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


def relax_pile(pile, site_order="forward"):
    """Relax one pile in place and return measurements of the event."""
    L = pile.shape[0]
    current = {tuple(site) for site in np.argwhere(pile >= 4)}
    topplings = 0
    toppled_sites = set()
    duration = 0

    while current:
        duration += 1
        ordered = sorted(current, reverse=(site_order == "reverse"))
        following = set()

        for x, y in ordered:
            pile[x, y] -= 4
            topplings += 1
            toppled_sites.add((x, y))

            if pile[x, y] >= 4:
                following.add((x, y))

            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                u, v = x + dx, y + dy
                if 0 <= u < L and 0 <= v < L:
                    pile[u, v] += 1
                    if pile[u, v] >= 4 and (u, v) not in current:
                        following.add((u, v))

        following.update({site for site in current if pile[site] >= 4})
        current = following

    return {
        "size": topplings,
        "area": len(toppled_sites),
        "duration": duration,
    }


def add_grain_and_relax(pile, rng, site=None, **relaxation_options):
    """Add one grain, relax the pile in place and return the event."""
    if site is None:
        site = tuple(rng.integers(0, pile.shape[0], size=2))
    pile[site] += 1
    event = relax_pile(pile, **relaxation_options)
    event["trigger"] = site
    return event


def prepare_active_pile(L=32, seed=3024):
    """Relax an overfull random state to obtain a stable starting pile."""
    rng = np.random.default_rng(seed)
    pile = rng.integers(0, 8, size=(L, L), dtype=np.int64)
    preparation = relax_pile(pile)
    return pile, preparation


def run_sandpile(L=32, additions=12_000, burn_in=2_000,
                 seed=3024, initial=None, progress=None):
    """Drive one pile and record every post-burn-in addition."""
    rng = np.random.default_rng(seed)
    pile = np.zeros((L, L), dtype=np.int64) if initial is None else initial.copy()
    sizes, areas, durations, mean_loads = [], [], [], []

    if progress is None:
        progress = additions >= 100_000
    addition_steps = range(additions)
    if progress:
        addition_steps = tqdm(
            addition_steps, total=additions, desc=f"Sandpile L={L}"
        )

    for step in addition_steps:
        event = add_grain_and_relax(pile, rng)
        if step >= burn_in:
            sizes.append(event["size"])
            areas.append(event["area"])
            durations.append(event["duration"])
            mean_loads.append(pile.mean())

    return {
        "pile": pile,
        "size": np.asarray(sizes),
        "area": np.asarray(areas),
        "duration": np.asarray(durations),
        "mean_load": np.asarray(mean_loads),
        "recorded_additions": additions - burn_in,
    }


def run_sandpile_ensemble(seeds, *, description="Independent sandpile runs",
                          **settings):
    """Run the same experiment with independent random seeds."""
    seeds = list(seeds)
    settings = dict(settings)
    settings.setdefault("progress", False)
    return [
        run_sandpile(seed=int(seed), **settings)
        for seed in tqdm(seeds, desc=description)
    ]

## Check the code you will use

Inspect individual updates, reproduce known cases, then check how the functions work together.

1. Show one of the piles below using `plt.imshow`, with a fixed colour scale and a colourbar. Pause before and after one complete toppling, using a debugger or saved copies inside `relax_pile`, and show both states. Follow the four grains and check the redistribution and boundary loss. Change a load of four to three to check the threshold.
2. Reproduce the reference cases below. Compare the complete final pile, $S$, $A_{\mathrm{av}}$, $T$ and grains lost. Explain the results. Compare forward and reverse processing, and choose further cases where you want more checks.
3. Trace one small avalanche. Show the pile after each parallel step and record which sites topple. Count all topplings, distinct sites and parallel steps, then compare those counts with the returned $S$, $A_{\mathrm{av}}$ and $T$. Check that added logging leaves the results unchanged.
4. Rebuild a short run using individual `add_grain_and_relax` calls. Compare every recorded field and the final pile with `run_sandpile`, using the same starting pile and seed. Include events with different $S$, $A_{\mathrm{av}}$ and $T$. Check that burn-in trials are omitted, no-toppling trials are included, repeating the seed repeats the results, and the supplied initial pile is unchanged. Compare addition sites with the seeded generator's draws.
5. Run a small ensemble. Compare each member with an individual `run_sandpile` call using its requested seed. Are different seeds producing different histories?

### Known small cases

These results follow directly from the toppling rules. All piles are $3\times3$; each list is one row. Make each starting pile with `np.array(..., dtype=int)`, then call `relax_pile` without adding a grain. Grains lost = starting load minus final load.

| Case | Starting pile | Final pile | $(S,A_{\mathrm{av}},T)$ | Grains lost |
|---|---|---|---|---|
| Interior | `[[0,0,0],[0,4,0],[0,0,0]]` | `[[0,1,0],[1,0,1],[0,1,0]]` | `(1,1,1)` | 0 |
| Edge | `[[0,4,0],[0,0,0],[0,0,0]]` | `[[1,0,1],[0,1,0],[0,0,0]]` | `(1,1,1)` | 1 |
| Corner | `[[4,0,0],[0,0,0],[0,0,0]]` | `[[0,1,0],[1,0,0],[0,0,0]]` | `(1,1,1)` | 2 |
| Repeated toppling | `[[0,0,0],[0,8,0],[0,0,0]]` | `[[0,2,0],[2,0,2],[0,2,0]]` | `(2,1,2)` | 0 |
| Two sites in one step | `[[0,0,0],[0,4,4],[0,0,0]]` | `[[0,1,1],[1,1,1],[0,1,1]]` | `(2,2,1)` | 1 |

Use assertions to repeat your checks. If a result disagrees, follow the relevant update and find the cause before running the investigation.

### Assertion reminder

`assert` stops the cell if a condition is false. Use the reference values or counts reconstructed from your trace:

```python
assert event["size"] == expected_size, "Unexpected toppling count"
assert np.array_equal(pile, expected_pile), "Unexpected final pile"
```

`np.array_equal` compares whole arrays; `pile == expected_pile` gives one result per site.

In [41]:
# Inspect updates, reproduce the reference cases and reconstruct event counts. Use assertions.

# Run the investigation

Does increasing lattice width extend the observed range of avalanche sizes?

## Investigation

Choose a confidence level and precision target using ‘How many runs are enough?’ before the full run.

Use $L=16,24,32$ and start with 20 independent seeds at each width. Start empty, use $8L^2$ additions as burn-in, then record 1,500 trials. Keep zeros and store each run separately. Compare mean load in the first and second halves of each record. If load still drifts consistently, extend burn-in and rerun, as in Week 6.

Make three plots:

1. Time series: show $S_n$ for four runs at $L=32$ over the same 500 recorded trials.
2. Run summaries: calculate each run's 95th percentile of non-zero avalanche sizes—the size exceeded by about 5% of its avalanches. Plot every run at each $L$, with the ensemble mean and your chosen confidence interval from `mean_interval`.
3. Distributions: pool non-zero avalanches at each $L$ and plot their CCDFs on common log–log axes.

The time series show when large avalanches occur. The run summaries show spread between independent runs. The CCDFs show how often larger avalanches occur.

### Supplied analysis functions

The CCDF gives the probability of reaching or exceeding a size. This helper removes zeros, so it gives $\Pr(S\geq s\mid S>0)$. Check it with zeros and repeated sizes. Keep zeros in the time series.

`mean_interval` gives approximate confidence bounds for the mean. Supply one summary per independent run.

In [42]:
def empirical_ccdf(values):
    values = np.sort(np.asarray(values))
    values = values[values > 0]
    unique, first = np.unique(values, return_index=True)
    probability = (len(values) - first) / len(values)
    return unique, probability


def mean_interval(values, confidence=0.90, draws=2_000, seed=3024):
    """Approximate confidence bounds for the mean of independent run summaries."""
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or values.size == 0 or not np.all(np.isfinite(values)):
        raise ValueError("Supply one finite value per independent run")
    if not 0 < confidence < 1:
        raise ValueError("Confidence must be between 0 and 1")
    if not isinstance(draws, (int, np.integer)) or draws < 1:
        raise ValueError("The number of draws must be a positive integer")
    rng = np.random.default_rng(seed)
    resampled = rng.choice(values, size=(draws, len(values)), replace=True)
    means = resampled.mean(axis=1)
    alpha = (1 - confidence) / 2
    return np.quantile(means, [alpha, 1 - alpha])


## Before the full run

Run one short pilot. Check the records, runtime, final stability and record count. Then run the investigation.

In [43]:
# Run one pilot, then the initial 20 runs at each L. Keep runs separate.

# Analyse


In [44]:
# Make the time-series, run-level and CCDF comparisons.

## How many runs are enough?

Count independent runs. Successive avalanches within one run can be related.

We are estimating the mean of the run-level 95th percentiles. Use `mean_interval` for its approximate confidence bounds. Express the full interval width as a percentage of the ensemble mean:

$$\text{relative interval width (\%)}=100\frac{\text{upper bound}-\text{lower bound}}{|\text{ensemble mean}|}.$$

Choose the confidence level and useful precision before running ensembles; keep these fixed across $L$. The instructor example uses 90% confidence and a 5% full-width target. Plot the percentage against run count, with a horizontal target line.

For a mean, width typically falls as $R^{-1/2}$, where $R$ counts independent runs. Halving the width takes about four times as many runs. See [Law (2015), §2 and §5.1](https://informs-sim.org/wsc15papers/188.pdf).

1. What precision is useful for your finite-size comparison? Why?
2. Which sizes meet your target at 20 runs? Can the mean look stable while its interval is still too wide?
3. Add 10 fresh seeds. How do the mean and interval width change? Does the interval still meet your target?
4. With confidence, relative target and record length fixed, how many runs do you need at each $L$? How does spread between runs, relative to their mean, help explain the difference?
5. At each $L$, choose a size in the far CCDF tail from the initial ensemble and keep it fixed. Count the avalanches reaching it and the independent runs contributing them. How do these counts change as you add runs?
6. Does a narrow interval for the mean run-level percentile also tell you the largest avalanches are well sampled? Which quantity does your target check?
7. For a few identical seeds at one $L$, compare 1,500 and 3,000 recorded trials with the same burn-in. How much do their percentiles change?

Add batches of 10 while the interval is too wide. When it first meets the target, add another 10 to check that it still does. Stop at 60 runs per $L$; report any target still unmet or not yet confirmed. Give both the first count meeting the target and the total after checking it. These counts apply to this quantity and protocol.

Check loading drift and record length separately. More independent runs can narrow the interval without fixing either problem.

In [45]:
# Compare 20 runs with the next batch against a fixed target; inspect far-tail contributions.

## Optional: further down the ladder

Choose another result and work back to its updates or data:

- Count grains sent beyond the boundary. For a trial starting from a stable pile, check

  $$\text{load before}+1=\text{load after}+\text{boundary loss}.$$

- Reconstruct one CCDF point by counting the non-zero avalanches reaching its threshold. How many independent runs contribute?

Add logging where needed. Check that it leaves the final pile and measurements unchanged.

## Interpret the comparison

Does the upper range of avalanche sizes extend as $L$ increases? Is the difference large relative to spread between runs?

# Conclusions


## Thinking about your project

Choose one result or figure from your project.

- Which rules and measurements have you checked? Can you trace the result back to individual updates?
- Could initialisation, burn-in or record length still affect it?
- What precision do you need? How many independent runs provide it, and does that change with domain size or parameters?
- If your claim concerns rare outcomes, how many events and independent runs support it?

Which check would most increase your confidence in this result?